# 02 · 시선 잔차 모델 학습·비교·ONNX 내보내기

기존 엔진(현재 보정 파이프라인 전체)을 첫 baseline 으로 두고, 같은 분할·같은 feature 계약·같은 평가 집합에서 **① 기존 엔진 그대로 · ② ridge 잔차 · ③ 작은 MLP 잔차**를 비교합니다.

- 잔차 target = 신뢰할 수 있는 표적 좌표 − baseline. baseline 은 Node 로 **실제 엔진을 재현**해 만듭니다(Python 재작성 없음 → parity drift 없음). 정밀 보정 8점은 ‘선택용 모델’ 기준으로만 독립 평가입니다.
- 분할은 학습 전에 고정(group/time). 표준화·feature 선택은 train 에서만 학습. early stopping·선택은 validation 만 사용. locked test 는 release gate 에서 1회 읽고 접근 기록을 남깁니다.
- 라벨 수집 모드의 마지막 holdout 블록은 eval_only(보고 전용) 입니다.
- 데이터가 작으면 ridge 가 이길 수 있고, 신경망이라는 이유로 교체하지 않습니다. 정책(`release_policy.json`)을 통과한 후보만 **candidate** 로 등록되며 active 가 아닙니다.

In [ ]:
#@title ① 환경 (01 과 같은 셀 — 새 런타임이면 다시 실행)
import os, subprocess, sys
if not os.path.isdir('/content/neurolens'): subprocess.check_call(['git', 'clone', '--depth', '50', 'https://github.com/jeebs0627/neurolens.git', '/content/neurolens'])
os.chdir('/content/neurolens'); sys.path.insert(0, '/content/neurolens/tools/colab')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'tools/colab/requirements-colab.txt'])
if r.returncode != 0: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'scipy', 'scikit-learn', 'onnx', 'onnxruntime']); subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', '--index-url', 'https://download.pytorch.org/whl/cpu'])
from google.colab import drive  # type: ignore
drive.mount('/content/drive'); RAW = '/content/drive/MyDrive/neurolens-training/raw'; WORK = '/content/drive/MyDrive/neurolens-training/work'
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '(GPU 는 필수가 아닙니다)')

In [ ]:
#@title ② 학습 실행 (체크포인트: 끊기면 같은 셀 재실행 → 이어서)
from nlcolab.pipeline import Pipeline
from nlcolab.safe_io import load_json
pl = Pipeline(RAW, WORK, use_mlp=True)
inv = load_json(f'{WORK}/inventory.json')  # 01 에서 만든 인벤토리
snap, diff, trigger, reason = pl.snapshot(inv)
if not trigger:
    print('학습하지 않음:', reason)
else:
    rep = pl.gaze(inv, snap)
    print('상태:', rep['status'], '· 선택:', rep['chosen'], rep['why'])
    print(open(f'{WORK}/gaze/REPORT.md', encoding='utf-8').read())

In [ ]:
#@title ③ (선택) ablation — 입력 feature 부분집합 비교는 validation 만 사용
# from nlcolab import gaze_dataset, train_gaze
# 예: head pose 제외 / 양안·단안 / weak click 포함 여부. 결과는 gaze/ablation-*.json 으로 저장하고 test 를 보고 feature 를 고치지 않는다.

## 결과 해석
- `status = insufficient_data`: 적격 세션·그룹·창이 정책 최소량보다 적습니다. 수집(특히 `/condition?mode=gaze-label`)을 늘리세요. 기존 엔진이 그대로 active 입니다.
- `status = evaluated`: 학습·평가는 끝났지만 정책을 통과하지 못했거나(실패 항목 표시) baseline 이 더 나았습니다. active 변경 없음.
- `status = candidate`: `work/gaze/candidate/models/gaze/<id>/` 에 ONNX·manifest·모델 카드, `model-registry.candidate.json` 에 registry 패치가 생성됐습니다 → `04_release_candidate.ipynb`.

오차는 **표적 proxy 기준**(CSS px · 화면 폭 %)이며 외부 시선 추적기 정확도가 아닙니다. 시거리를 재지 않았으므로 시각도(°)로 보고하지 않습니다.